## Purely serial execution

- No multithreading
- No multiprocessing

In [ ]:
def _prep_temperature_grid_and_rmatrix(tresp, tresp_logt, temps, emd_int=False):
    """
    Recreate the dn2dem_pos internals to get logt, dlogt, and rmatrix
    (single-thread, no pools). Matches dn2dem_pos logic.               :contentReference[oaicite:1]{index=1}
    """
    # Bin centers in log10 space
    dlogt = (np.log10(temps[1:]) - np.log10(temps[:-1]))
    nt = len(dlogt)
    logt = np.array([np.log10(temps[0]) + (dlogt[i] * (i + 0.5)) for i in range(nt)])
    nf = tresp.shape[1]

    # Clean/clip tresp like dn2dem_pos does
    truse = np.zeros_like(tresp)
    for i in range(nf):
        good = tresp[:, i] > 0
        truse[good, i] = tresp[good, i]
        truse[~good, i] = np.min(tresp[good, i])

    # Interp response in log-space (matches dn2dem_pos)                :contentReference[oaicite:2]{index=2}
    tr = np.zeros((nt, nf))
    for i in range(nf):
        tr[:, i] = 10 ** np.interp(logt, tresp_logt, np.log10(truse[:, i]))

    # Build rmatrix (includes 1/K factor and scaling)                  :contentReference[oaicite:3]{index=3}
    dlogTfac = 10.0 ** logt * np.log(10.0 ** dlogt)
    rmatrix = np.zeros((nt, nf))
    if emd_int:
        rmatrix[:, :] = tr
    else:
        for i in range(nf):
            rmatrix[:, i] = tr[:, i] * dlogTfac

    sclf = 1e15
    rmatrix *= sclf
    return logt, dlogt, rmatrix, sclf

def _solve_pixel_serial(dn_px, edn_px, rmatrix, logt, dlogt, reg_tweak=1.0, max_iter=10,
                        rgt_fact=1.5, dem_norm0=None, nmu=42, l_emd=False, rscl=False):
    """
    Single pixel DEM using dem_pix (serial, no pools).                 :contentReference[oaicite:4]{index=4}
    """
    nf = rmatrix.shape[1]
    if dem_norm0 is None:
        dem_norm0 = np.ones(len(logt))
    glc = np.zeros(nf, dtype=int)  # no EM-loci weighting by default

    dem, edem, elogt, chisq, dn_reg = dem_pix(
        dn_px, edn_px, rmatrix, logt, dlogt, glc,
        reg_tweak=reg_tweak, max_iter=max_iter, rgt_fact=rgt_fact,
        dem_norm0=dem_norm0, nmu=nmu, warn=False, l_emd=l_emd, rscl=rscl
    )
    return dem, edem, elogt, chisq, dn_reg

def temperature_logmap_from_frame(frame_dn, tresp, tresp_logt, temps,
                                  edn_frame=None, reg_tweak=1.0, max_iter=10,
                                  rgt_fact=1.5, nmu=42, emd_int=False, l_emd=False,
                                  rscl=False):
    """
    Compute log-temperature maps for a single frame (6,H,W) in a single thread/process.

    Returns:
      logT_emw (H,W): emission-measure–weighted log10(T)   (base-10)
      logT_peak (H,W): log10(T) at DEM peak               (base-10)
    """
    if frame_dn.ndim != 3:
        raise ValueError("frame_dn must have shape (nf, H, W).")

    nf, H, W = frame_dn.shape
    if edn_frame is None:
        # Simple Poisson-ish error: sqrt(dn) (avoid zeros)
        edn_frame = np.sqrt(np.clip(frame_dn, 0, None)) + 1e-6

    # Prepare temperature grid + response matrix (matches dn2dem_pos)  :contentReference[oaicite:5]{index=5}
    logt, dlogt, rmatrix, sclf = _prep_temperature_grid_and_rmatrix(
        tresp, tresp_logt, temps, emd_int=emd_int
    )
    T_centers = 10 ** logt  # Kelvin

    # Output maps
    logT_emw = np.full((H, W), np.nan, dtype=np.float64)
    logT_peak = np.full((H, W), np.nan, dtype=np.float64)

    # Linear, serial loop over pixels
    for y in tqdm(range(H), desc="Rows", unit="row"):
        for x in range(W):
            dn_px = frame_dn[:, y, x].astype(np.float64)
            edn_px = edn_frame[:, y, x].astype(np.float64)

            # Skip empty/invalid pixels quickly
            if not np.isfinite(dn_px).all() or np.all(dn_px <= 0):
                continue

            dem, _, _, _, _ = _solve_pixel_serial(
                dn_px, edn_px, rmatrix, logt, dlogt,
                reg_tweak=reg_tweak, max_iter=max_iter,
                rgt_fact=rgt_fact, dem_norm0=None,
                nmu=nmu, l_emd=l_emd, rscl=rscl
            )

            # Undo the scale used inside rmatrix construction (to stay consistent
            # with dn2dem_pos returning DEM * sclf). Here dem already matches that,
            # so for temperature *ratios* the scale cancels; we don't need to rescale.

            # EM-weighted temperature (Kelvin) then to log10
            dem_pos = np.clip(dem, 0, None)
            s = dem_pos.sum()
            if s > 0:
                T_emw = np.dot(T_centers, dem_pos) / s
                logT_emw[y, x] = np.log10(T_emw)
                logT_peak[y, x] = logt[np.argmax(dem_pos)]
            # else leave NaN

    return logT_emw, logT_peak

def temperature_logmaps_from_stack(stack, tresp, tresp_logt, temps, edn_stack=None, **kwargs):
    """
    Apply temperature_logmap_from_frame to every frame in a stack
    shaped (N, nf, H, W) OR (nf, H, W). Single-threaded outer loop.
    Returns:
      logT_emw_all: (N, H, W)  or (H, W) if input had no N
      logT_peak_all: (N, H, W) or (H, W)
    """
    if stack.ndim == 3:
        # Single frame
        return temperature_logmap_from_frame(stack, tresp, tresp_logt, temps,
                                             edn_frame=edn_stack, **kwargs)
    elif stack.ndim == 4:
        N, nf, H, W = stack.shape
        if edn_stack is not None and edn_stack.shape != stack.shape:
            raise ValueError("edn_stack must match stack shape.")
        logT_emw_all = np.empty((N, H, W), dtype=np.float64)
        logT_peak_all = np.empty((N, H, W), dtype=np.float64)
        for i in tqdm(range(N), desc="Frames", unit="frame"):  # serial over frames
            frame = stack[i]
            edn_frame = None if edn_stack is None else edn_stack[i]
            emw, peak = temperature_logmap_from_frame(
                frame, tresp, tresp_logt, temps, edn_frame=edn_frame, **kwargs
            )
            logT_emw_all[i] = emw
            logT_peak_all[i] = peak
        return logT_emw_all, logT_peak_all
    else:
        raise ValueError("stack must be (nf,H,W) or (N,nf,H,W).")


In [ ]:
# Example (one frame):
logT_emw, logT_peak = temperature_logmap_from_frame(
    frame_dn=STACK[0],
    tresp=T_RESP,
    tresp_logt=T_RESP_LOGT,
    temps=TEMPS,
    reg_tweak=1.0,
    max_iter=10,
    rgt_fact=1.5,
    nmu=42,
    emd_int=False,   # set True to invert in EMD space if you prefer
    l_emd=False,
    rscl=False
)

## Calculate all stacks

In [ ]:
def stack_to_temp_maps(STACK, T_RESP, T_RESP_LOGT, TEMPS, nmu=42):
    """
    Run dn2dem_pos on each frame in STACK and return
    mean- and peak-logT maps of shape (N,H,W).
    """
    mean_maps, peak_maps = [], []

    for i in tqdm(range(STACK.shape[0]), desc="Frames", unit="frame"):
        frame = np.moveaxis(STACK[i], 0, -1).astype(np.float64, copy=False)
        frame[~np.isfinite(frame)] = 0.0
        frame = np.clip(frame, 0, None)

        edn = np.sqrt(frame) + 1e-6
        edn[~np.isfinite(edn)] = 1e-6

        demmap, edemmap, logT_bins, chisq, dn_reg = dn2dem_pos(
            frame, edn, T_RESP, T_RESP_LOGT, TEMPS, nmu=nmu
        )

        dem = np.clip(demmap, 0, None).astype(np.float32, copy=False)
        EM = dem.sum(axis=2); valid = EM > 0
        imax = dem.argmax(axis=2).astype(np.intp)

        peak = np.where(valid, np.take(np.asarray(logT_bins).reshape(-1), imax), np.nan)
        T_centers = (10.0**np.asarray(logT_bins)).astype(np.float32)
        num = np.einsum('ijk,k->ij', dem, T_centers, optimize=True)
        mean = np.full(EM.shape, np.nan, dtype=np.float32)
        np.divide(num, EM, out=mean, where=valid)
        mean = np.log10(mean, out=mean, where=valid)

        mean_maps.append(mean); peak_maps.append(peak)

    return np.stack(mean_maps), np.stack(peak_maps)

mean_maps, peak_maps = stack_to_temp_maps(STACK, T_RESP, T_RESP_LOGT, TEMPS, nmu=42)